<a href="https://colab.research.google.com/github/intisariapps-com/intiVoice_Studio/blob/main/intiVoice_Studio_WebUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎙️ intiVoice AI — Web Studio Visual Mandiri (Gradio Edition V1.6.3)

> ⏰ **Terakhir Diperbarui:** 21 September 2026 | **Rilis:** V1.6.3 (Standalone 1-Click Studio)
> 🖼️ **Interactive Inline UI:** Antarmuka visual interaktif langsung tampil di bawah sel Colab (*embedded iframe*).
> 🌐 **Public URL Tunnel Otomatis:** Menghasilkan link publik acak gratis (`https://xxxx.gradio.live`) tanpa perlu akun Cloudflare atau token.
> 🚀 **Zero-PopUp Ultra-Fast Startup:** Memasang dependensi via `uv` (~3 detik) dan memuat model langsung ke NVMe SSD Colab.

Notebook ini adalah **Antarmuka Mandiri (All-in-One)** untuk Text-to-Speech (TTS), Voice Design, dan Voice Cloning berkualitas **48kHz Studio Audio**.

### ✨ Fitur Utama:
1. 🎨 **Preset Karakter Bahasa Indonesia**: Suara wanita lembut, pria narator wibawa, podcast santai, dan pembaca berita formal.
2. 🎙️ **Ultimate & Controllable Voice Cloning**: Cukup unggah rekaman suara acuan (WAV/MP3) untuk meniru warna vokal siapapun.
3. 🎛️ **Voice Design Deskriptif**: Bentuk karakter vokal baru menggunakan instruksi alami di dalam tanda kurung `(A warm female voice)`.
4. ⚙️ **Kontrol Parameter Ahli**: Slider CFG Scale (kepatuhan naskah), Kecepatan Bicara (Native Speed), dan Seed DNA.
5. 🔊 **Pemutar Audio Langsung & Download**: Putar hasil suara 48kHz di dalam Colab dan unduh berkas WAV hanya dengan 1 klik.

---
### 🚀 Cara Menjalankan:
1. Pastikan runtime GPU aktif (**Runtime** → **Change runtime type** → **T4 GPU**).
2. Klik tombol **Play (▶)** pada sel kode di bawah ini.
3. Tunggu ~1 menit hingga model siap. Antarmuka Web Studio interaktif akan otomatis muncul di bawah sel, dan link publik `https://xxxx.gradio.live` akan tercetak di layar.


In [ ]:
"""
🎙️ INTIVOICE AI — STANDALONE 1-CLICK GRADIO STUDIO (V1.6.3)
Hak Cipta (C) 2026 IntisariApps.com. Seluruh hak cipta dilindungi.
"""

# @title 🚀 LUNCURKAN WEB STUDIO VISUAL (1-KLIK)
# @markdown Jalankan sel ini untuk memulai Web Studio interaktif di layar Colab dan membuat link publik gratis.

import os
import sys
import time
import subprocess

print("=" * 80)
print("🎙️ MEMULAI INTIVOICE AI — STANDALONE GRADIO STUDIO V1.6.3")
print("=" * 80)

# 1. Verifikasi GPU
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    print("⚠️ PERINGATAN: Runtime GPU tidak terdeteksi!")
    print("👉 Buka menu 'Runtime' -> 'Change runtime type' -> Pilih 'T4 GPU' lalu jalankan ulang.")
else:
    print("✅ Akselerator GPU Aktif & Siap Digunakan.")

# 2. Instalasi Dependensi Cepat via uv (~3-5 detik)
print("\n[1/3] ⚡ Memeriksa & memasang dependensi (Ultra-Fast via uv)...")
try:
    import voxcpm
    import soundfile as sf
    import gradio as gr
    import torch
    import torchaudio
    print("✅ Seluruh pustaka audio & model sudah aktif di memori!")
except ImportError:
    print("⏳ Memasang voxcpm, soundfile, gradio, torchaudio via uv...")
    try:
        subprocess.check_call(["uv", "pip", "install", "--system", "voxcpm", "soundfile", "gradio", "ipython", "torchaudio", "librosa"])
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "voxcpm", "soundfile", "gradio", "ipython", "torchaudio", "librosa"])
    
    import voxcpm
    import soundfile as sf
    import gradio as gr
    import torch
    import torchaudio
    print("✅ Dependensi berhasil dipasang sempurna!")

from voxcpm import VoxCPM

# 3. Muat Model VoxCPM2 ke GPU
print("\n[2/3] 🧠 Memuat Model VoxCPM2 ke GPU...")
start_load = time.time()
model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
print(f"✅ Model VoxCPM2 siap di GPU dalam {time.time() - start_load:.2f} detik!")

# 4. Preset Karakter Bahasa Indonesia
PRESETS = {
    "🌸 Wanita Lembut & Ramah (Customer Service / Storytelling)": "A young Indonesian woman, gentle, sweet and very friendly voice, clear and soothing tone",
    "🎙️ Pria Narator Berwibawa (Dokumenter / E-Learning / Iklan)": "A mature Indonesian male narrator, deep warm authoritative voice, calm and professional pacing",
    "🎧 Host Podcast Santai & Energik (YouTube / TikTok / Sosmed)": "An energetic young Indonesian male podcast host, cheerful, expressive and casual tone",
    "📢 Pembaca Berita Resmi (News Anchor / Formal)": "A professional female news anchor, formal Indonesian intonation, clear articulation and confident pacing",
    "⚡ Kustom (Ketik Prompt Karakter Sendiri)": ""
}

def synthesize_studio(text, preset_choice, custom_prompt, ref_audio, ref_transcript, cfg_value, steps, seed, speed):
    if not text or not text.strip():
        return None, "⚠️ Silakan masukkan naskah teks terlebih dahulu!"

    # Tentukan gaya vokal
    if preset_choice == "⚡ Kustom (Ketik Prompt Karakter Sendiri)":
        style = custom_prompt.strip()
    else:
        style = PRESETS.get(preset_choice, "")

    # Gabungkan speed hint alami
    speed_hint = ""
    if speed < 0.90:
        speed_hint = "speak slowly and clearly"
    elif speed > 1.15:
        speed_hint = "speak fast and briskly"

    controls = []
    if style:
        controls.append(style)
    if speed_hint:
        controls.append(speed_hint)

    prompt_prefix = f"({', '.join(controls)})" if controls else ""
    full_text = f"{prompt_prefix}{text.strip()}"

    # Atur Seed DNA
    if seed is not None and seed >= 0:
        torch.manual_seed(int(seed))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(seed))

    start_gen = time.time()

    # Konfigurasi inferensi
    gen_kwargs = {
        "text": full_text,
        "cfg_value": float(cfg_value),
        "inference_timesteps": int(steps),
        "retry_badcase": True,
    }

    if ref_audio is not None:
        gen_kwargs["reference_wav_path"] = ref_audio
        if ref_transcript and ref_transcript.strip():
            gen_kwargs["prompt_wav_path"] = ref_audio
            gen_kwargs["prompt_text"] = ref_transcript.strip()

    with torch.inference_mode():
        wav = model.generate(**gen_kwargs)

    sample_rate = getattr(model.tts_model, "sample_rate", 48000)
    duration = len(wav) / sample_rate
    elapsed = time.time() - start_gen
    rtf = elapsed / duration if duration > 0 else 0

    os.makedirs("/tmp/intivoice_outputs", exist_ok=True)
    out_path = f"/tmp/intivoice_outputs/audio_{int(time.time())}.wav"
    sf.write(out_path, wav, sample_rate)

    mode_name = "Ultimate Clone 1:1" if (ref_audio and ref_transcript) else ("Timbre Clone" if ref_audio else "Voice Design")
    status_msg = (
        f"✨ Sintesis Berhasil [{mode_name}] | "
        f"Waktu Render: {elapsed:.2f}s | Durasi Audio: {duration:.2f}s (48kHz) | RTF: {rtf:.2f}"
    )
    return out_path, status_msg

# 5. Bangun Tampilan Gradio Web Studio (Dark/Light Modern Theme)
print("\n[3/3] 🌐 Meluncurkan Web Studio Publik...")

custom_theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="cyan",
    neutral_hue="slate",
).set(
    button_primary_background_fill="*primary_500",
    button_primary_background_fill_hover="*primary_600",
)

with gr.Blocks(theme=custom_theme, title="intiVoice Studio Web") as demo:
    gr.Markdown(
        """
        # 🎙️ intiVoice AI — Studio Suara Pintar (48kHz Studio Audio)
        ### Sintesis Suara Ekspresif, Voice Design & Ultimate Voice Cloning 1:1
        *Jalankan langsung di Google Colab atau buka melalui tautan publik acak di bawah.*
        """
    )

    with gr.Row():
        # Kolom Kiri: Input Naskah & Pengaturan
        with gr.Column(scale=5):
            text_input = gr.Textbox(
                label="📝 Naskah Teks (Bahasa Indonesia / Multilingual)",
                placeholder="Ketik atau tempel naskah teks Anda di sini...",
                lines=6,
                value="Halo semuanya! Selamat datang di era baru kecerdasan buatan, di mana setiap cerita dan suara memiliki kehangatan yang nyata."
            )

            with gr.Tabs():
                with gr.TabItem("🎨 Preset Karakter"):
                    preset_dropdown = gr.Dropdown(
                        choices=list(PRESETS.keys()),
                        value=list(PRESETS.keys())[0],
                        label="Pilih Karakter Suara Resmi"
                    )
                    custom_prompt_input = gr.Textbox(
                        label="Kustom Deskripsi Karakter Suara",
                        placeholder="Contoh: A cheerful young Indonesian mother, soft and warm tone...",
                        lines=2,
                        visible=False
                    )

                with gr.TabItem("🎙️ Ultimate Voice Cloning"):
                    gr.Markdown("Unggah rekaman vokal 6-8 detik (WAV 48kHz ideal) untuk meniru timbre dan karakter vokal asli.")
                    ref_audio_input = gr.Audio(
                        label="Audio Referensi Kloning",
                        type="filepath"
                    )
                    ref_transcript_input = gr.Textbox(
                        label="Naskah Transkrip Audio Referensi (Wajib untuk Kloning 1:1 Presisi)",
                        placeholder="Ketik kata-kata persis yang diucapkan pada file audio referensi di atas...",
                        lines=2
                    )

            with gr.Accordion("⚙️ Parameter Vokal & Performa (Tuning)", open=False):
                with gr.Row():
                    speed_slider = gr.Slider(minimum=0.75, maximum=1.35, value=1.0, step=0.05, label="Kecepatan Bicara (Speed)")
                    cfg_slider = gr.Slider(minimum=1.0, maximum=4.0, value=2.0, step=0.1, label="CFG Scale (Kepatuhan Naskah)")
                with gr.Row():
                    steps_slider = gr.Slider(minimum=5, maximum=25, value=10, step=1, label="Inference Timesteps")
                    seed_input = gr.Number(value=428190, label="Fixed Seed (DNA Karakter Suara)")

            btn_submit = gr.Button("🚀 Sintesis Suara 48kHz Sekarang", variant="primary", size="lg")

        # Kolom Kanan: Player Hasil Audio
        with gr.Column(scale=4):
            audio_output = gr.Audio(
                label="🔊 Hasil Audio Sintesis (WAV 48.000 Hz)",
                type="filepath",
                interactive=False
            )
            status_output = gr.Textbox(
                label="📊 Status Sintesis & Metrik Waktu",
                interactive=False
            )
            gr.Markdown(
                """
                > [!TIP]
                > **Cara Mengunduh Berkas:**
                > Klik ikon titik tiga (⋮) pada pemutar audio di atas, lalu klik **Unduh / Download** untuk menyimpan audio WAV 48kHz beresolusi penuh.
                """
            )

    def update_custom_visibility(choice):
        return gr.update(visible=(choice == "⚡ Kustom (Ketik Prompt Karakter Sendiri)"))

    preset_dropdown.change(fn=update_custom_visibility, inputs=[preset_dropdown], outputs=[custom_prompt_input])

    btn_submit.click(
        fn=synthesize_studio,
        inputs=[
            text_input,
            preset_dropdown,
            custom_prompt_input,
            ref_audio_input,
            ref_transcript_input,
            cfg_slider,
            steps_slider,
            seed_input,
            speed_slider,
        ],
        outputs=[audio_output, status_output]
    )

print("\n================================================================================")
print("🚀 Antarmuka Web Studio Siap! Link Publik (gradio.live) akan muncul di bawah:")
print("================================================================================")
demo.queue().launch(share=True, inline=True)
